# 📰 Fake News Detection using Graph ML
## Notebook 3: GNN Model Training

**Models we train and compare:**
| Model | Key idea |
|-------|----------|
| **GCN** | Averages neighbor features (spectral convolution) |
| **GAT** | Learns *attention weights* over neighbors |
| **GraphSAGE** | Samples + aggregates neighbors (scales to large graphs) |

**Task:** Binary node classification — Fake (1) vs Real (0)

## Step 1 — Setup

In [ ]:
!pip install torch-geometric scikit-learn matplotlib -q
print('✅ Ready!')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle, os

from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')

## Step 2 — Load Graph Data

In [ ]:
graph_data = torch.load('data/graph_data.pt')
with open('data/meta.pkl', 'rb') as f:
    meta = pickle.load(f)

graph_data = graph_data.to(device)

IN_DIM  = graph_data.num_node_features
OUT_DIM = 2  # binary: fake vs real

print(f'Graph loaded to {device}')
print(f'Input feature dim : {IN_DIM}')
print(f'Nodes: {graph_data.num_nodes:,} | Edges: {graph_data.num_edges:,}')
print(f'Train: {graph_data.train_mask.sum().item():,} | Val: {graph_data.val_mask.sum().item():,} | Test: {graph_data.test_mask.sum().item():,}')

## Step 3 — Define GNN Models

All three models follow the same 2-layer structure:
```
Input (feat_dim) → GNN Layer 1 (128) → ReLU → Dropout → GNN Layer 2 (64) → Linear → Softmax
```

In [ ]:
# ── GCN ───────────────────────────────────────────────────────────────────────
class GCN(nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=2, dropout=0.5):
        super().__init__()
        self.conv1   = GCNConv(in_dim, hidden)
        self.conv2   = GCNConv(hidden, hidden // 2)
        self.linear  = nn.Linear(hidden // 2, out_dim)
        self.dropout = dropout

    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        x = self.linear(x)
        return F.log_softmax(x, dim=1), x  # (logits, embeddings)

# ── GAT ───────────────────────────────────────────────────────────────────────
class GAT(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=2, heads=4, dropout=0.5):
        super().__init__()
        self.conv1  = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
        self.conv2  = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout)
        self.linear = nn.Linear(hidden, out_dim)
        self.dropout = dropout

    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv2(x, ei))
        x = self.linear(x)
        return F.log_softmax(x, dim=1), x

# ── GraphSAGE ─────────────────────────────────────────────────────────────────
class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=2, dropout=0.5):
        super().__init__()
        self.conv1  = SAGEConv(in_dim, hidden)
        self.conv2  = SAGEConv(hidden, hidden // 2)
        self.linear = nn.Linear(hidden // 2, out_dim)
        self.dropout = dropout

    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        x = self.linear(x)
        return F.log_softmax(x, dim=1), x

print('✅ GCN, GAT, GraphSAGE defined!')

## Step 4 — Training Loop

In [ ]:
def train_model(model, data, epochs=100, lr=0.01, weight_decay=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_state   = None

    model.train()
    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        optimizer.zero_grad()
        out, _ = model(data)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        scheduler.step()

        # ── Validate ──
        model.eval()
        with torch.no_grad():
            out, _ = model(data)
            val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask]).item()
            pred = out.argmax(dim=1)

            t_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            v_acc = (pred[data.val_mask]   == data.y[data.val_mask]).float().mean().item()

        history['train_loss'].append(loss.item())
        history['val_loss'].append(val_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 20 == 0:
            print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f}')

    model.load_state_dict(best_state)  # restore best weights
    print(f'\n🏆 Best Val Acc: {best_val_acc:.4f}')
    return history

print('✅ Training function ready!')

## Step 5 — Train All Three Models

In [ ]:
# ── GCN ───────────────────────────────────────────────────────────────────────
print('=' * 50)
print('Training GCN...')
print('=' * 50)
gcn_model = GCN(in_dim=IN_DIM).to(device)
gcn_history = train_model(gcn_model, graph_data, epochs=100)

In [ ]:
# ── GAT ───────────────────────────────────────────────────────────────────────
print('=' * 50)
print('Training GAT...')
print('=' * 50)
gat_model = GAT(in_dim=IN_DIM).to(device)
gat_history = train_model(gat_model, graph_data, epochs=100)

In [ ]:
# ── GraphSAGE ─────────────────────────────────────────────────────────────────
print('=' * 50)
print('Training GraphSAGE...')
print('=' * 50)
sage_model = GraphSAGE(in_dim=IN_DIM).to(device)
sage_history = train_model(sage_model, graph_data, epochs=100)

## Step 6 — Evaluate on Test Set

In [ ]:
def evaluate(model, data, name='Model'):
    model.eval()
    with torch.no_grad():
        out, embeddings = model(data)
        pred  = out.argmax(dim=1)
        probs = torch.exp(out)[:, 1]  # probability of fake

    mask = data.test_mask
    y_true  = data.y[mask].cpu().numpy()
    y_pred  = pred[mask].cpu().numpy()
    y_prob  = probs[mask].cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_prob)

    print(f'\n{" " + name + " Test Results ":=^50}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'  AUC-ROC  : {auc:.4f}')
    print(f'\n{classification_report(y_true, y_pred, target_names=["Real", "Fake"])}')

    return {'name': name, 'acc': acc, 'f1': f1, 'auc': auc,
            'embeddings': embeddings[:meta['n_articles']].cpu().numpy(),
            'y_true': y_true, 'y_pred': y_pred}

gcn_results  = evaluate(gcn_model,  graph_data, 'GCN')
gat_results  = evaluate(gat_model,  graph_data, 'GAT')
sage_results = evaluate(sage_model, graph_data, 'GraphSAGE')

In [ ]:
# ── Model comparison bar chart ─────────────────────────────────────────────────
models   = ['GCN', 'GAT', 'GraphSAGE']
acc_vals = [gcn_results['acc'],  gat_results['acc'],  sage_results['acc']]
f1_vals  = [gcn_results['f1'],   gat_results['f1'],   sage_results['f1']]
auc_vals = [gcn_results['auc'],  gat_results['auc'],  sage_results['auc']]

x = np.arange(len(models))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w,   acc_vals, w, label='Accuracy', color='#3498db')
ax.bar(x,       f1_vals,  w, label='F1 Score', color='#2ecc71')
ax.bar(x + w,   auc_vals, w, label='AUC-ROC',  color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=13)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('GNN Model Comparison — Test Set')
ax.legend()

for i, (a, f, u) in enumerate(zip(acc_vals, f1_vals, auc_vals)):
    ax.text(i - w,   a + 0.01, f'{a:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i,       f + 0.01, f'{f:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + w,   u + 0.01, f'{u:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, label, color in [
    (gcn_history,  'GCN',       '#3498db'),
    (gat_history,  'GAT',       '#e74c3c'),
    (sage_history, 'GraphSAGE', '#2ecc71'),
]:
    axes[0].plot(hist['val_loss'], label=label, color=color)
    axes[1].plot(hist['val_acc'],  label=label, color=color)

axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('NLL Loss')
axes[0].legend()

axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Training Curves — All Models', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Save Models & Results

In [ ]:
os.makedirs('models', exist_ok=True)
torch.save(gcn_model.state_dict(),  'models/gcn.pt')
torch.save(gat_model.state_dict(),  'models/gat.pt')
torch.save(sage_model.state_dict(), 'models/sage.pt')

results_df = pd.DataFrame([gcn_results, gat_results, sage_results])[['name', 'acc', 'f1', 'auc']]
results_df.to_csv('results/metrics.csv', index=False)

# Save embeddings for visualization
with open('data/results.pkl', 'wb') as f:
    pickle.dump({'gcn': gcn_results, 'gat': gat_results, 'sage': sage_results}, f)

print('✅ Models saved to models/')
print('✅ Results saved to results/')
print(results_df.to_string(index=False))
print('\n🚀 Ready for Notebook 4: Visualization!')